# Chapter 6. Fuzzy-Set Qualitative Comparative Analysis

*Starting With What You Have: A Quantitative Field Guide for Urban Research in Data-Scarce Settings*

Runs in a browser with no installation. Open in Google Colab and choose Runtime, then Run all.


## Step 0. Installation

No fsQCA package exists for Python, so this book publishes `fsqca.py`. Place it and `data/` alongside this notebook.

The implementation has been cross-validated against the R `QCA` package and agrees to three decimal places.

In [ ]:
!pip install -q pandas numpy matplotlib

## Step 1. Fix the cases and conditions

32 cases with four conditions gives 16 corners. With $k$ conditions there are $2^k$ corners, so adding conditions costs diversity exponentially.

In [ ]:
import pandas as pd
import numpy as np
import fsqca

df = pd.read_csv("data/upgrading_cities.csv")
CONDS = ["tenure", "finance", "participation", "capacity"]
print("cases:", len(df), "| conditions:", len(CONDS), "| corners:", 2 ** len(CONDS))
df.head()

## Step 2. Calibration, the most important decision

Three anchors: full membership, crossover, full non-membership. **These come from theory, not from the sample distribution.**

In [ ]:
cond = {c: fsqca.calibrate(df[c], full_in=8, crossover=5, full_out=2) for c in CONDS}
Y = fsqca.calibrate(df["upgrade_success"], full_in=8, crossover=5, full_out=2)

pd.DataFrame({**cond, "outcome": Y}).head().round(4)

## The calibration curve

0.5 is the point of maximum ambiguity: the difference between 0.49 and 0.51 is qualitatively larger than that between 0.51 and 0.90.

In [ ]:
import matplotlib.pyplot as plt

raw = np.linspace(0, 10, 200)
plt.figure(figsize=(6, 4))
plt.plot(raw, fsqca.calibrate(raw, 8, 5, 2), lw=2)
for v, lab in [(2, "full out"), (5, "crossover"), (8, "full in")]:
    plt.axvline(v, ls="--", c="red", alpha=0.6)
    plt.text(v + 0.1, 0.05, lab, fontsize=9, color="red")
plt.xlabel("raw value")
plt.ylabel("set membership")
plt.grid(alpha=0.25)
plt.show()

## Step 3. Necessity analysis, before sufficiency

Necessity conventionally demands 0.90 or above. If no single condition clears it, that is already a finding: it rejects single-lever prescriptions.

In [ ]:
fsqca.necessity_table(cond, Y).round(3)

## Step 4. Build the truth table

Consistency is computed over **all cases weighted by their membership in each corner**, not over the assigned cases alone. Restricting to assigned cases is a common implementation error.

In [ ]:
tt = fsqca.truth_table(cond, Y, freq_cutoff=1, cons_cutoff=0.80)
tt.sort_values("consistency", ascending=False).head(8)

## Step 5. Boolean minimisation

The complex solution refuses logical remainders and is conservative. The parsimonious solution uses them freely. **Report both.**

In [ ]:
complex_sol = fsqca.minimise(tt, CONDS)
parsi_sol = fsqca.minimise(tt, CONDS, use_remainders=True)

def show(imp):
    return "*".join((CONDS[i] if v == "1" else "~" + CONDS[i])
                    for i, v in enumerate(imp) if v != "-")

print("complex:")
for p in complex_sol:
    print("   ", show(p))
print("parsimonious:")
for p in parsi_sol:
    print("   ", show(p))

## Step 6. Consistency and coverage for each path

A path with low unique coverage overlaps substantially with the others and should carry less weight in interpretation.

In [ ]:
mat = np.vstack([np.asarray(cond[c], float) for c in CONDS]).T
Yv = np.asarray(Y, float)

def membership(imp):
    m = np.ones(len(Yv))
    for i, v in enumerate(imp):
        if v == "1":
            m = np.minimum(m, mat[:, i])
        elif v == "0":
            m = np.minimum(m, 1 - mat[:, i])
    return m

rows = []
for p in complex_sol:
    m = membership(p)
    rows.append({"path": show(p),
                 "consistency": round(float(np.minimum(m, Yv).sum() / m.sum()), 3),
                 "coverage": round(float(np.minimum(m, Yv).sum() / Yv.sum()), 3)})
pd.DataFrame(rows)

## Step 7. XY plot

Under sufficiency, points sit **above** the diagonal. Under necessity they sit below it. The two are geometric opposites.

In [ ]:
best = max(complex_sol, key=lambda p: membership(p).sum())
m = membership(best)

plt.figure(figsize=(5.5, 5.5))
plt.scatter(m, Yv, s=45, alpha=0.8)
plt.plot([0, 1], [0, 1], "--", c="grey")
plt.xlabel(f"membership in {show(best)}")
plt.ylabel("outcome membership")
plt.title("Sufficiency: points above the diagonal")
plt.grid(alpha=0.25)
plt.show()

## Step 8. Robustness checks, which cannot be skipped

Raising the frequency threshold drops paths supported by only one case each, which tells you those paths are fragile. Do the same for the calibration crossover.

In [ ]:
for fc in [1, 2, 3]:
    t = fsqca.truth_table(cond, Y, freq_cutoff=fc, cons_cutoff=0.80)
    sol = fsqca.minimise(t, CONDS, use_remainders=True)
    print(f"freq_cutoff={fc}: " + " + ".join(show(p) for p in sol))

---

**What to do next.** Compare against Section 6.4. If you need the **intermediate solution**, which journals often expect, use fsQCA 4.x or R's `QCA`.